![FLIP Banner](../../Assets/images/flip-banner.png)

# FLIP: Agentic AI in Practice
**Module 08: Advanced Agentic AI**

---

## Session 8E: Agent Hooks and Workflow Guards

<div align="center">

<table>
<thead><tr><th><strong>Item</strong></th><th><strong>Description</strong></th></tr></thead>
<tbody>
<tr><td align="left">Estimated time</td><td>2 hours</td></tr>
<tr><td align="left">Mandatory part</td><td>Local hook engine for pre-tool, post-tool and stop events</td></tr>
<tr><td align="left">Main output</td><td>A controlled hook system that blocks unsafe calls and audits tool use</td></tr>
</tbody>
</table>

</div>

### 1. Overview and Learning Goals

A hook is a function or command that runs automatically at a specific workflow event. In agentic AI, hooks can guard tool use, inspect outputs, log actions or run final checks.

```mermaid
flowchart LR
    A[User request] --> B[Agent decides tool]
    B --> C[PreToolUse hook]
    C -->|allowed| D[Tool runs]
    C -->|blocked| E[Refusal]
    D --> F[PostToolUse hook]
    F --> G[Final response]
    G --> H[Stop hook]
```

Hooks are not a replacement for careful tool design. They are an additional control layer.

### 2. Conceptual Background

Common hook events:

<div align="center">

<table>
<thead><tr><th><strong>Hook event</strong></th><th><strong>When it runs</strong></th><th><strong>Example use</strong></th></tr></thead>
<tbody>
<tr><td align="left">PreToolUse</td><td>Before a tool runs.</td><td>Block shell commands or private-file access.</td></tr>
<tr><td align="left">PostToolUse</td><td>After a tool runs.</td><td>Log result metadata or check output format.</td></tr>
<tr><td align="left">Stop</td><td>Before workflow finishes.</td><td>Run final safety or audit checks.</td></tr>
</tbody>
</table>

</div>

A hook should be simple and predictable. It should check a concrete rule and return a concrete decision.

In [ ]:
import json
from dataclasses import dataclass, field
from typing import Any, Callable, Dict, List

print("M08E hooks setup complete.")

In [ ]:
@dataclass
class HookDecision:
    allowed: bool
    reason: str
    metadata: Dict[str, Any] = field(default_factory=dict)

@dataclass
class ToolCall:
    tool_name: str
    args: Dict[str, Any]

@dataclass
class AuditRecord:
    event: str
    tool_name: str
    allowed: bool
    reason: str
    metadata: Dict[str, Any] = field(default_factory=dict)

In [ ]:
class HookEngine:
    def __init__(self):
        self.hooks = {"PreToolUse": [], "PostToolUse": [], "Stop": []}
        self.audit_log: List[AuditRecord] = []

    def register(self, event: str, hook_fn: Callable[[Dict[str, Any]], HookDecision]) -> None:
        if event not in self.hooks:
            raise ValueError(f"Unknown hook event: {event}")
        self.hooks[event].append(hook_fn)

    def run_event(self, event: str, payload: Dict[str, Any]) -> HookDecision:
        if event not in self.hooks:
            return HookDecision(False, f"Unknown hook event: {event}")

        for hook_fn in self.hooks[event]:
            decision = hook_fn(payload)
            tool_name = payload.get("tool_name", "none")
            self.audit_log.append(AuditRecord(event, tool_name, decision.allowed, decision.reason, decision.metadata))
            if not decision.allowed:
                return decision

        return HookDecision(True, f"All {event} hooks passed.")

In [ ]:
def pre_tool_safety_hook(payload: Dict[str, Any]) -> HookDecision:
    tool_name = payload.get("tool_name", "")
    args = payload.get("args", {})
    blocked_tools = {"shell", "send_email", "read_private_file", "access_credentials"}

    if tool_name in blocked_tools:
        return HookDecision(False, f"Blocked unsafe tool: {tool_name}", {"blocked_tool": tool_name})

    arg_text = json.dumps(args).lower()
    blocked_terms = ["password", "api_key", "credential", "private file", "student record", "hidden solution"]

    if any(term in arg_text for term in blocked_terms):
        return HookDecision(False, "Blocked arguments containing sensitive or private terms.", {"args": args})

    return HookDecision(True, "Pre-tool safety check passed.")

def post_tool_audit_hook(payload: Dict[str, Any]) -> HookDecision:
    return HookDecision(True, "Post-tool audit recorded.", {"result_type": type(payload.get("result")).__name__})

def stop_review_hook(payload: Dict[str, Any]) -> HookDecision:
    return HookDecision(True, "Workflow stopped with audit summary.", {"blocked_calls": payload.get("blocked_calls", 0)})

In [ ]:
def rectangle_area(width: float, height: float) -> float:
    return width * height

def safe_word_count(text: str) -> int:
    return len(text.split())

def uppercase_tool(args):
    return str(args["text"]).upper()

TOOLS = {
    "rectangle_area": lambda args: rectangle_area(float(args["width"]), float(args["height"])),
    "word_count": lambda args: safe_word_count(str(args["text"])),
    "uppercase": uppercase_tool,
}

class HookedToolRunner:
    def __init__(self, hook_engine: HookEngine, tools: Dict[str, Callable[[Dict[str, Any]], Any]]):
        self.hook_engine = hook_engine
        self.tools = tools
        self.blocked_calls = 0

    def run_tool(self, tool_call: ToolCall) -> Dict[str, Any]:
        pre = self.hook_engine.run_event("PreToolUse", {"tool_name": tool_call.tool_name, "args": tool_call.args})
        if not pre.allowed:
            self.blocked_calls += 1
            return {"ok": True, "result": {"status": "blocked", "reason": pre.reason}}

        if tool_call.tool_name not in self.tools:
            self.blocked_calls += 1
            return {"ok": False, "error": f"Unknown tool: {tool_call.tool_name}", "result": None}

        try:
            result = self.tools[tool_call.tool_name](tool_call.args)
        except Exception as exc:
            return {"ok": False, "error": f"Tool execution failed: {exc}", "result": None}

        post = self.hook_engine.run_event("PostToolUse", {"tool_name": tool_call.tool_name, "args": tool_call.args, "result": result})
        return {"ok": True, "result": {"status": "completed", "tool_result": result, "post_hook": post.reason}}

    def stop(self) -> Dict[str, Any]:
        decision = self.hook_engine.run_event("Stop", {"blocked_calls": self.blocked_calls})
        return {"ok": decision.allowed, "result": {"summary": decision.reason, "blocked_calls": self.blocked_calls}}

In [ ]:
engine = HookEngine()
engine.register("PreToolUse", pre_tool_safety_hook)
engine.register("PostToolUse", post_tool_audit_hook)
engine.register("Stop", stop_review_hook)

runner = HookedToolRunner(engine, TOOLS)
print(runner.run_tool(ToolCall("rectangle_area", {"width": 3, "height": 4})))
print(runner.run_tool(ToolCall("shell", {"command": "rm -rf /"})))
print(runner.run_tool(ToolCall("uppercase", {"text": "safe hook"})))
print(runner.stop())

### 6. Inspection and Audit Logs

A hook system should keep audit records. The audit log should show what was checked, what was allowed, what was blocked and why.

In [ ]:
def display_audit_log(engine: HookEngine) -> None:
    for record in engine.audit_log:
        print(f"{record.event} | tool={record.tool_name} | allowed={record.allowed} | reason={record.reason}")
        if record.metadata:
            print("  metadata:", record.metadata)

display_audit_log(engine)

### 7. Optional Coding-Agent Hook Mapping

The local hook engine maps conceptually to coding-agent hooks:

<div align="center">

<table>
<thead><tr><th><strong>Local hook</strong></th><th><strong>Coding-agent use</strong></th></tr></thead>
<tbody>
<tr><td align="left">PreToolUse</td><td>Block unsafe shell commands or protected-file edits.</td></tr>
<tr><td align="left">PostToolUse</td><td>Log modified files or run formatting checks.</td></tr>
<tr><td align="left">Stop</td><td>Run final scan or remind user to review changes.</td></tr>
</tbody>
</table>

</div>

In [ ]:
print("Optional real hook-system configuration is not required for this lab.")

### 8. Testing and Analysis

In [ ]:
test_engine = HookEngine()
test_engine.register("PreToolUse", pre_tool_safety_hook)
test_engine.register("PostToolUse", post_tool_audit_hook)
test_engine.register("Stop", stop_review_hook)
test_runner = HookedToolRunner(test_engine, TOOLS)

allowed = test_runner.run_tool(ToolCall("rectangle_area", {"width": 6, "height": 7}))
assert allowed["ok"] is True
assert allowed["result"]["status"] == "completed"
assert allowed["result"]["tool_result"] == 42

blocked_tool = test_runner.run_tool(ToolCall("send_email", {"to": "x@example.com"}))
assert blocked_tool["ok"] is True
assert blocked_tool["result"]["status"] == "blocked"

blocked_args = test_runner.run_tool(ToolCall("word_count", {"text": "my password is 123"}))
assert blocked_args["ok"] is True
assert blocked_args["result"]["status"] == "blocked"

uppercase = test_runner.run_tool(ToolCall("uppercase", {"text": "safe"}))
assert uppercase["ok"] is True
assert uppercase["result"]["tool_result"] == "SAFE"

unknown = test_runner.run_tool(ToolCall("unknown_tool", {}))
assert unknown["ok"] is False

stop = test_runner.stop()
assert stop["ok"] is True
assert stop["result"]["blocked_calls"] >= 2

assert len(test_engine.audit_log) >= 4

print("All M08E mandatory hook tests passed.")

### 9. Student Tasks

<div align="center">

<table>
<thead><tr><th><strong>Task</strong></th><th><strong>What to do</strong></th><th><strong>Evidence</strong></th></tr></thead>
<tbody>
<tr><td align="left">Run baseline tests</td><td>Run all mandatory cells.</td><td>Test output.</td></tr>
<tr><td align="left">Add a new PreToolUse rule</td><td>Block one additional unsafe argument pattern.</td><td>Hook code.</td></tr>
<tr><td align="left">Add a new safe tool</td><td>For example, <code>lowercase(text)</code>.</td><td>Tool code and registry update.</td></tr>
<tr><td align="left">Add tests</td><td>Valid call, blocked call, stop hook and audit check.</td><td>Assert-based tests.</td></tr>
<tr><td align="left">Analyse audit log</td><td>Explain allowed and blocked actions.</td><td>Short paragraph.</td></tr>
</tbody>
</table>

</div>

### 10. Submission and Reflection

Submit:

```text
1. Baseline test output.
2. New hook rule.
3. New safe tool.
4. Added tests.
5. Audit-log analysis.
6. 150–250 word reflection.
```

Further readings:

- Claude Code hooks reference: https://code.claude.com/docs/en/hooks
- Claude Code hooks guide: https://code.claude.com/docs/en/hooks-guide
- MCP official introduction: https://modelcontextprotocol.io/docs/getting-started/intro
- LangGraph documentation: https://langchain-ai.github.io/langgraph/